# Databricks PySpark Transformations
Goal: Transform raw airline CSV into curated fact & dimension tables using PySpark + Delta Lake.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType
from pyspark.sql.functions import col, to_date, year, month
from delta.tables import DeltaTable

## Checking keyvault

In [0]:
dbutils.secrets.listScopes()

[SecretScope(name='kv-airline-lab')]

In [0]:
dbutils.secrets.get(
  scope="kv-airline-lab",
  key="kvdatalakepktestlab"
)

'[REDACTED]'

In [0]:
#Legacy (Cluster-scoped config, works with your current cluster)
#format = spark.conf.set("fs.azure.account.key.[storage account name].dfs.core.windows.net", dbutils.secrets.get(scope="[scope name]", key="[key vault key name]"))")
spark.conf.set(
  "fs.azure.account.key.datalakepktestlab.dfs.core.windows.net",
  dbutils.secrets.get(scope="kv-airline-lab", key="kvdatalakepktestlab")
)

In [0]:
#test list the directory using dbutils.fs.ls("abfss://[container name]@[storage account name].dfs.core.windows.net]/[folder]/[subfolder1]/[subfolder2]...")
dbutils.fs.ls(
    "abfss://raw@datalakepktestlab.dfs.core.windows.net/airline_dw/2024/"
)

[FileInfo(path='abfss://raw@datalakepktestlab.dfs.core.windows.net/airline_dw/2024/01/', name='01/', size=0, modificationTime=1766164422000),
 FileInfo(path='abfss://raw@datalakepktestlab.dfs.core.windows.net/airline_dw/2024/02/', name='02/', size=0, modificationTime=1767722055000),
 FileInfo(path='abfss://raw@datalakepktestlab.dfs.core.windows.net/airline_dw/2024/03/', name='03/', size=0, modificationTime=1767722055000)]

##Connect to Source

In [0]:
raw_path = "abfss://raw@datalakepktestlab.dfs.core.windows.net/airline_dw/2024/"

df_raw = (
    spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .option("recursiveFileLookup", "true")
        .csv(raw_path, mode='DROPMALFORMED')
)

display(df_raw)

FlightDate,AirlineCode,AirlineName,FlightNumber,OriginAirport,OriginCity,OriginState,DestAirport,DestCity,DestState,ScheduledDepTime,ActualDepTime,DepDelayMinutes,ScheduledArrTime,ActualArrTime,ArrDelayMinutes,Cancelled,CancellationReason,DistanceKM
2024-02-01,AA,American Airlines,1001,JFK,New York,NY,LAX,Los Angeles,CA,800,820,20,1100,1140,40,0,null,3974
2024-02-03,DL,Delta Airlines,2034,ATL,Atlanta,GA,ORD,Chicago,IL,930,945,15,1100,1120,20,0,null,975
2024-02-05,UA,United Airlines,778,SFO,San Francisco,CA,SEA,Seattle,WA,700,655,-5,830,820,-10,0,null,1090
2024-02-08,AA,American Airlines,455,PHX,Phoenix,AZ,DFW,Dallas,TX,1300,1330,30,1630,1700,30,0,null,1390
2024-02-10,DL,Delta Airlines,445,DTW,Detroit,MI,DEN,Denver,CO,1200,1200,0,1400,1355,-5,0,null,1820
2024-02-12,UA,United Airlines,880,ORD,Chicago,IL,LAX,Los Angeles,CA,1400,1430,30,1630,1705,35,0,null,2805
2024-02-15,AA,American Airlines,300,JFK,New York,NY,BOS,Boston,MA,900,930,30,1015,1045,30,0,null,300
2024-02-18,DL,Delta Airlines,789,LAX,Los Angeles,CA,SEA,Seattle,WA,1600,1700,60,1830,1935,65,1,Mechanical,1545
2024-02-22,UA,United Airlines,901,DEN,Denver,CO,ATL,Atlanta,GA,1100,1105,5,1630,1635,5,0,null,1935
2024-02-25,AA,American Airlines,1099,JFK,New York,NY,MIA,Miami,FL,600,600,0,900,910,10,0,null,1757


#Data Cleaning

In [0]:

#Explicitly Cast Cancelled. Never rely on inferSchema for business columns.
df_clean = df_raw.withColumn(
    "Cancelled",                
    col("Cancelled").cast("int")   # OR boolean — choose ONE forever
)


###Casting data types

In [0]:
#Align All Columns Explicitly (Best Practice), casting data types
df_clean = (
    df_raw
        .withColumn("Cancelled", col("Cancelled").cast("int"))
        .withColumn("DepDelayMinutes", col("DepDelayMinutes").cast("int"))
        .withColumn("ArrDelayMinutes", col("ArrDelayMinutes").cast("int"))
        .withColumn("DistanceKM", col("DistanceKM").cast("int"))
)


###Remove Duplicates

In [0]:
#4️⃣ Data Cleansing & Standardization (Silver Rules), drop duplicates


df_clean = (
    df_raw
    .dropDuplicates([
        "FlightNumber",
        "FlightDate",
        "AirlineCode",
        "FlightNumber",
        "OriginAirport",
        "DestAirport"
    ])
)

display(df_clean)

FlightDate,AirlineCode,AirlineName,FlightNumber,OriginAirport,OriginCity,OriginState,DestAirport,DestCity,DestState,ScheduledDepTime,ActualDepTime,DepDelayMinutes,ScheduledArrTime,ActualArrTime,ArrDelayMinutes,Cancelled,CancellationReason,DistanceKM
2024-01-01,AA,American Airlines,1001,JFK,New York,NY,LAX,Los Angeles,CA,800,815,15,1100,1135,35,0,null,3974
2024-01-02,DL,Delta Airlines,2034,ATL,Atlanta,GA,ORD,Chicago,IL,930,925,-5,1100,1050,-10,0,null,975
2024-01-03,UA,United Airlines,778,SFO,San Francisco,CA,SEA,Seattle,WA,700,710,10,830,845,15,0,null,1090
2024-01-05,AA,American Airlines,1099,JFK,New York,NY,MIA,Miami,FL,600,600,0,900,855,-5,0,null,1757
2024-01-07,DL,Delta Airlines,445,DTW,Detroit,MI,DEN,Denver,CO,1200,1230,30,1400,1435,35,0,null,1820
2024-01-10,UA,United Airlines,880,ORD,Chicago,IL,LAX,Los Angeles,CA,1400,1500,60,1630,1740,70,0,null,2805
2024-01-15,AA,American Airlines,300,JFK,New York,NY,BOS,Boston,MA,900,905,5,1015,1020,5,0,null,300
2024-01-18,DL,Delta Airlines,789,LAX,Los Angeles,CA,SEA,Seattle,WA,1600,1600,0,1830,1825,-5,0,null,1545
2024-01-22,UA,United Airlines,901,DEN,Denver,CO,ATL,Atlanta,GA,1100,1130,30,1630,1700,30,0,null,1935
2024-01-25,AA,American Airlines,455,PHX,Phoenix,AZ,DFW,Dallas,TX,1300,1400,60,1630,1745,75,1,Weather,1390


#Creating Fact Table

###Add Columns for Year and Month

In [0]:
fact_flights_enriched = df_clean \
    .withColumn("FlightYear", year("FlightDate")) \
    .withColumn("FlightMonth", month("FlightDate"))


display(fact_flights_enriched)

FlightDate,AirlineCode,AirlineName,FlightNumber,OriginAirport,OriginCity,OriginState,DestAirport,DestCity,DestState,ScheduledDepTime,ActualDepTime,DepDelayMinutes,ScheduledArrTime,ActualArrTime,ArrDelayMinutes,Cancelled,CancellationReason,DistanceKM,FlightYear,FlightMonth
2024-01-01,AA,American Airlines,1001,JFK,New York,NY,LAX,Los Angeles,CA,800,815,15,1100,1135,35,0,null,3974,2024,1
2024-01-02,DL,Delta Airlines,2034,ATL,Atlanta,GA,ORD,Chicago,IL,930,925,-5,1100,1050,-10,0,null,975,2024,1
2024-01-03,UA,United Airlines,778,SFO,San Francisco,CA,SEA,Seattle,WA,700,710,10,830,845,15,0,null,1090,2024,1
2024-01-05,AA,American Airlines,1099,JFK,New York,NY,MIA,Miami,FL,600,600,0,900,855,-5,0,null,1757,2024,1
2024-01-07,DL,Delta Airlines,445,DTW,Detroit,MI,DEN,Denver,CO,1200,1230,30,1400,1435,35,0,null,1820,2024,1
2024-01-10,UA,United Airlines,880,ORD,Chicago,IL,LAX,Los Angeles,CA,1400,1500,60,1630,1740,70,0,null,2805,2024,1
2024-01-15,AA,American Airlines,300,JFK,New York,NY,BOS,Boston,MA,900,905,5,1015,1020,5,0,null,300,2024,1
2024-01-18,DL,Delta Airlines,789,LAX,Los Angeles,CA,SEA,Seattle,WA,1600,1600,0,1830,1825,-5,0,null,1545,2024,1
2024-01-22,UA,United Airlines,901,DEN,Denver,CO,ATL,Atlanta,GA,1100,1130,30,1630,1700,30,0,null,1935,2024,1
2024-01-25,AA,American Airlines,455,PHX,Phoenix,AZ,DFW,Dallas,TX,1300,1400,60,1630,1745,75,1,Weather,1390,2024,1


###Select column as fact table

In [0]:
fact_flights = fact_flights_enriched.select(
    "FlightNumber",
    "FlightDate",
    "AirlineCode",
    "OriginAirport",
    "DestAirport",
    "DepDelayMinutes",
    "ArrDelayMinutes",
    "Cancelled",
    "DistanceKM",
    "FlightYear",
    "FlightMonth"
).orderBy("FlightDate")

In [0]:
display(fact_flights)

FlightNumber,FlightDate,AirlineCode,OriginAirport,DestAirport,DepDelayMinutes,ArrDelayMinutes,Cancelled,DistanceKM,FlightYear,FlightMonth
1001,2024-01-01,AA,JFK,LAX,15,35,0,3974,2024,1
2034,2024-01-02,DL,ATL,ORD,-5,-10,0,975,2024,1
778,2024-01-03,UA,SFO,SEA,10,15,0,1090,2024,1
1099,2024-01-05,AA,JFK,MIA,0,-5,0,1757,2024,1
445,2024-01-07,DL,DTW,DEN,30,35,0,1820,2024,1
880,2024-01-10,UA,ORD,LAX,60,70,0,2805,2024,1
300,2024-01-15,AA,JFK,BOS,5,5,0,300,2024,1
789,2024-01-18,DL,LAX,SEA,0,-5,0,1545,2024,1
901,2024-01-22,UA,DEN,ATL,30,30,0,1935,2024,1
455,2024-01-25,AA,PHX,DFW,60,75,1,1390,2024,1


##Overwrite Delta log

In [0]:
%sql
DESCRIBE DETAIL delta.`abfss://enriched@datalakepktestlab.dfs.core.windows.net/airline_dw/fact_flights`

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,b8d35607-069f-428e-a616-02a990799a93,null,null,abfss://enriched@datalakepktestlab.dfs.core.windows.net/airline_dw/fact_flights,2025-12-25T15:22:09.742Z,2026-01-08T11:04:54Z,"List(FlightYear, FlightMonth)",List(),3,8866,Map(delta.enableDeletionVectors -> true),3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
DESCRIBE TABLE delta.`abfss://enriched@datalakepktestlab.dfs.core.windows.net/airline_dw/fact_flights`

--it shows that the data type for Cancelled column is string the destination table while in the databricks spark table is integer. 

col_name,data_type,comment
FlightNumber,int,null
FlightDate,date,null
AirlineCode,string,null
OriginAirport,string,null
DestAirport,string,null
DepDelayMinutes,int,null
ArrDelayMinutes,int,null
Cancelled,int,null
DistanceKM,int,null
FlightYear,int,null


###DISABLING Deletion Vector 

In [0]:

spark.conf.set("spark.databricks.delta.properties.defaults.enableDeletionVectors", "false")


##MERGE
### Pointing to Enriched folder / fact_flights subfolder

In [0]:
#Merge. I removed the flight_number since it was not in the list of matching columns
delta_path = "abfss://enriched@datalakepktestlab.dfs.core.windows.net/airline_dw/fact_flights"

delta_table = DeltaTable.forPath(spark, delta_path)

delta_table.alias("t").merge(
    fact_flights.alias("s"),
    """
    t.FlightDate = s.FlightDate
    AND t.AirlineCode = s.AirlineCode
    AND t.FlightNumber = s.FlightNumber
    AND t.OriginAirport = s.OriginAirport
    AND t.DestAirport = s.DestAirport
    """
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()


###Use Overwrite mode for emergency

In [0]:
#Overwrite all regardless of data type since error is encountered on  data types/ Should be careful.
fact_flights.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("abfss://enriched@datalakepktestlab.dfs.core.windows.net/airline_dw/fact_flights")


#Dimension Tables

##DimAirline — SCD Type 1 (Recommended)

###AIRLINE

In [0]:
#Step 1: Re-derive Dimension DataFrames (YES)
dim_airline_src = (
    df_clean
        .select("AirlineCode", "AirlineName")
        .dropDuplicates(["AirlineCode"])
)

In [0]:
# Step 2: Check If Delta Table Exists (CRITICAL)
# from delta.tables import DeltaTable

dim_airline_path = "abfss://enriched@datalakepktestlab.dfs.core.windows.net/airline_dw/dim_airline"

if DeltaTable.isDeltaTable(spark, dim_airline_path):
    print("DimAirline exists — MERGE")
else:
    print("DimAirline does not exist — CREATE")


DimAirline exists — MERGE


In [0]:
display(dim_airline_src)

AirlineCode,AirlineName
AA,American Airlines
DL,Delta Airlines
UA,United Airlines


In [0]:
#from delta.tables import DeltaTable -> already imported. 
#dimension table exists, use the code below: 

dim_delta = DeltaTable.forPath(spark, dim_airline_path)

dim_delta.alias("t").merge(
    dim_airline_src.alias("s"),
    "t.AirlineCode = s.AirlineCode"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()



###Use Overwrite mode regarding disregarding Deletion Vector for Synapse

In [0]:
##Error encountered in Synapse (Feature 'Deletion Vectors' is not supported for table 'AirlineLake/airline_dw/dim_airport'.)
#Solution: Overwrite the dimension tables AND Disable
# This didn't work, synapse still find the deletion vector, need to create a clean container 

dim_airline_src.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.enableDeletionVectors", "false") \
    .save("abfss://enriched@datalakepktestlab.dfs.core.windows.net/airline_dw/dim_airline")


In [0]:
#Create a clean container

dim_airline_src.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.enableDeletionVectors", "false") \
    .save("abfss://enriched@datalakepktestlab.dfs.core.windows.net/airline_dw/dim_airline_clean")


###AIRPORT CODE

In [0]:
#Step 1: Re-derive DimAirport (Source DF)
#“Airport dimension is handled as SCD Type 1 using Delta MERGE on AirportCode.”
dim_airport_src = (
    df_clean
        .select(
            col("OriginAirport").alias("AirportCode"),
            col("OriginCity").alias("City"),
            col("OriginState").alias("State")
        )
        .union(
            df_clean.select(
                col("DestAirport").alias("AirportCode"),
                col("DestCity").alias("City"),
                col("DestState").alias("State")
            )
        )
        .dropDuplicates(["AirportCode"])
)


In [0]:
display(dim_airport_src)

AirportCode,City,State
ATL,Atlanta,GA
BOS,Boston,MA
DEN,Denver,CO
DFW,Dallas,TX
DTW,Detroit,MI
JFK,New York,NY
LAX,Los Angeles,CA
MIA,Miami,FL
ORD,Chicago,IL
PHX,Phoenix,AZ


In [0]:
# 🔹 Step 2: MERGE Into Delta

dim_airport_path = "abfss://enriched@datalakepktestlab.dfs.core.windows.net/airline_dw/dim_airport"

if DeltaTable.isDeltaTable(spark, dim_airport_path):
    dim_airport_delta = DeltaTable.forPath(spark, dim_airport_path)

    dim_airport_delta.alias("t").merge(
        dim_airport_src.alias("s"),
        "t.AirportCode = s.AirportCode"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
else:
    dim_airport_src.write.format("delta") \
        .mode("overwrite") \
        .save(dim_airport_path)


In [0]:
##Error encountered in Synapse (Feature 'Deletion Vectors' is not supported for table 'AirlineLake/airline_dw/dim_airport'.)
#Solution: Overwrite the dimension tables AND Disable 

dim_airport_src.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.enableDeletionVectors", "false") \
    .save("abfss://enriched@datalakepktestlab.dfs.core.windows.net/airline_dw/dim_airport")


In [0]:
#Create a clean container

dim_airport_src.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.enableDeletionVectors", "false") \
    .save("abfss://enriched@datalakepktestlab.dfs.core.windows.net/airline_dw/dim_airport_clean")


###Dim Date

In [0]:
#this is just getting the dates in the source file. 
from pyspark.sql.functions import (
    to_date, year, month, dayofmonth, weekofyear, quarter
)

dim_date = (
    df_clean
        .select(to_date("FlightDate").alias("Date"))
        .distinct()
        .withColumn("Year", year("Date"))
        .withColumn("Month", month("Date"))
        .withColumn("Day", dayofmonth("Date"))
        .withColumn("Week", weekofyear("Date"))
        .withColumn("Quarter", quarter("Date"))
)

display(dim_date)

Date,Year,Month,Day,Week,Quarter
2024-02-05,2024,2,5,6,1
2024-02-15,2024,2,15,7,1
2024-02-08,2024,2,8,6,1
2024-02-10,2024,2,10,6,1
2024-02-25,2024,2,25,8,1
2024-02-12,2024,2,12,7,1
2024-02-01,2024,2,1,5,1
2024-02-03,2024,2,3,5,1
2024-02-18,2024,2,18,7,1
2024-02-22,2024,2,22,8,1


In [0]:
from pyspark.sql.functions import min, max

dates = spark.read.format("delta") \
    .load("abfss://enriched@datalakepktestlab.dfs.core.windows.net/airline_dw/fact_flights")

start_date = dates.selectExpr("min(FlightDate)").collect()[0][0]
end_date   = dates.selectExpr("max(FlightDate)").collect()[0][0]


display(start_date)
display(end_date)

datetime.date(2024, 1, 1)

datetime.date(2024, 3, 27)

In [0]:
from pyspark.sql.functions import explode, sequence, to_date, lit, expr

dim_date_dynamic = (
    spark.sql("SELECT 1")
    .withColumn(
        "Date",
        explode(
            sequence(
                to_date(lit(start_date)),
                to_date(lit(end_date)),
                expr("interval 1 day")
            )
        )
    )
)

display(dim_date_dynamic)

1,Date
1,2024-01-01
1,2024-01-02
1,2024-01-03
1,2024-01-04
1,2024-01-05
1,2024-01-06
1,2024-01-07
1,2024-01-08
1,2024-01-09
1,2024-01-10


###DimDate will be done to Synapse